In [0]:
# --- 1. SET CONTEXT ---
spark.sql("USE CATALOG healthcare_p360")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

from pyspark.sql.functions import col, cast, when, current_timestamp

# --- 2. CLEAN & TRANSFORM PATIENTS ---
df_patients = spark.table("bronze.patients") \
    .withColumn("date_of_birth", col("date_of_birth").cast("date")) \
    .withColumn("ingested_at", current_timestamp())

# --- 3. CLEAN & TRANSFORM LABS ---
# Adding logic for the abnormal_flag
df_labs = spark.table("bronze.labs") \
    .withColumn("test_date", col("test_date").cast("date")) \
    .withColumn("is_abnormal", when(col("abnormal_flag") == "Y", True).otherwise(False))

# --- 4. JOIN TABLES (The Unified View) ---
# We join Patients with Labs and Diagnoses to see everything in one place
df_silver_unified = df_patients.join(df_labs, "patient_id", "left") \
    .join(spark.table("bronze.diagnoses"), "patient_id", "left")

# --- 5. SAVE TO SILVER ---
df_silver_unified.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.unified_patient_health_record")

print("✅ Silver Layer transformation complete: unified_patient_health_record created.")